# Day-1 Cockpit (version 2, canonical)

Four levels, in run order. You never write code; you run cells with Shift+Enter and read sentences.

| Level | What it is | Where it lives |
|---|---|---|
| 1 | Ready answers, no student data needed | `context_pack/CONTEXT_FINDINGS.md`, events file, drafts in `docs/` |
| 2 | Insights from the Day-1 data + hypothesis verdicts | CELLS 1-10, then the VERDICTS cell |
| 3 | Why a place or cohort stands where it stands | `outputs/dossiers/` (one per district + GENDER.md) |
| 4 | Assembly and submission | Final cell + `playbook/DAY1_ALGORITHM.md` |

If a cell errors: read `outputs/RUN_LOG.txt`, follow the failure branch in the algorithm. Never debug.


In [35]:
# CELL 1 - where am I, what data do I have
import os, glob, pandas as pd, json
BASE=r'C:\Users\CSF\Desktop\Datathon\datathon2026'
if os.path.exists(BASE): os.chdir(BASE)   # anchor, regardless of where Jupyter started
print('Working in:', os.getcwd())
pd.set_option('display.width',200); pd.set_option('display.max_colwidth',90)
assert os.path.exists('src/run_all.py'), 'Folder moved? Fix BASE in this cell.'
files=[f for f in glob.glob('data/primary/*.csv') if 'SYNTHETIC' not in f.upper()]
print('Real data files found:', len(files))
for f in files: print('  %s  %.0f MB'%(f, os.path.getsize(f)/1e6))
if not files: print('No real files: tonight that is correct; CELL 2 will run in rehearsal mode.')


Working in: C:\Users\CSF\Desktop\Datathon\datathon2026
Real data files found: 9
  data/primary\std_grade4_2022-23.csv  14 MB
  data/primary\std_grade4_2023-24.csv  21 MB
  data/primary\std_grade4_2024-25.csv  28 MB
  data/primary\std_grade5_2022-23.csv  15 MB
  data/primary\std_grade5_2023-24.csv  21 MB
  data/primary\std_grade5_2024-25.csv  27 MB
  data/primary\std_grade6_2022-23.csv  13 MB
  data/primary\std_grade6_2023-24.csv  20 MB
  data/primary\std_grade6_2024-25.csv  24 MB


In [36]:
# CELL 2 - RUN THE WHOLE PIPELINE (mode-aware)
import shutil
real=[f for f in glob.glob('data/primary/*.csv') if 'SYNTHETIC' not in f.upper()]
if real:
    print('SATURDAY MODE - real data found:', len(real), 'file(s)')
    for f in glob.glob('data/primary/SYNTHETIC*.csv'):
        shutil.move(f, f+'.bak'); print('  parked', f)
else:
    print('REHEARSAL MODE - no real data, using synthetic files')
    for f in glob.glob('data/primary/SYNTHETIC_2*.csv.bak'):
        shutil.move(f, f[:-4]); print('  restored', f[:-4])
    if not glob.glob('data/primary/SYNTHETIC*.csv'):
        print('  regenerating synthetic files (one minute)')
        !python prep/03_make_synthetic.py
print('\nStarting pipeline...')
!python src/run_all.py


SATURDAY MODE - real data found: 9 file(s)

Starting pipeline...
DATATHON 2026 PIPELINE   seed=20260801

[  0.0s] LOAD PRIMARY DATA
   files stacked: 9 | competency columns built: 11
      + std_grade4_2024-25.csv (209867 rows, 28 MB)
      + std_grade5_2024-25.csv (204008 rows, 27 MB)
      + std_grade6_2024-25.csv (181653 rows, 24 MB)
      + std_grade5_2023-24.csv (161979 rows, 21 MB)
      + std_grade4_2023-24.csv (156940 rows, 21 MB)
      + std_grade6_2023-24.csv (151938 rows, 20 MB)
      + std_grade5_2022-23.csv (113164 rows, 15 MB)
      + std_grade4_2022-23.csv (104015 rows, 14 MB)
      + std_grade6_2022-23.csv (95523 rows, 13 MB)

   --- column detection (from first file) ---
   year      <- Year                         
   grade     <- Grade                        
   division  <- NOT FOUND                      <-- check config.MANUAL_OVERRIDES
   district  <- District                     
   block     <- Block                        
   cluster   <- Cluster               

In [22]:
# CELL 3 - data quality: read this before believing anything
display(pd.read_csv('outputs/tables/qa_flags.csv'))
display(pd.read_csv('outputs/tables/qa_summary.csv').head(30))


,severity,issue
0,HIGH,Contest COVERAGE changes across years (not a naming problem: the organiser crosswalk r...


,check,value,note
0,rows,1379087,NaN
1,question_items_found,20,handbook says 20
2,exact_duplicate_rows,84951,identical on geography + all item responses
3,duplicate_rows_note,"EXPECTED, not an error: no persistent student or school identifier exists, so two chil...",NaN
4,missing_pct__year,0.0,NaN
5,missing_pct__grade,0.0,NaN
6,missing_pct__district,0.0,NaN
7,missing_pct__block,0.0,NaN
8,missing_pct__cluster,0.0,NaN
9,missing_pct__gp,0.0,NaN


Every HIGH flag becomes a limitation sentence. L2 types each one into the report Doc now, while you read them aloud.


In [23]:
# CELL 4 - coverage: who actually got tested
# At DISTRICT level: the only unit where our geography and UDISE match 100%.
# GP-level coverage is NOT computed - GP names do not join reliably (see the gate, 37%).
import numpy as np, sys
sys.path.insert(0,'src'); import external
u=pd.read_csv('outputs/tables/unit_district_by_year.csv')
den=pd.read_csv('external_data/udise_karnataka_gp_grade46_enrolment.csv')
al=external.build_alias_table()
md=external.match_districts(den['district'].dropna().unique(), al, 'district')
den=den.merge(md[['district','canonical_district']], on='district', how='left')
d2=den.groupby(['canonical_district','academic_year'])['enrol_g4_6_govt'].sum().reset_index()
mu=external.match_districts(u['district'].dropna().unique(), al, 'district')
u2=u.merge(mu[['district','canonical_district']], on='district', how='left')
m=u2.merge(d2, left_on=['canonical_district','year'], right_on=['canonical_district','academic_year'], how='left')
m['coverage_pct']=100*m['n_students']/m['enrol_g4_6_govt'].replace(0,np.nan)
ok=m['coverage_pct'].notna()
print('district-years with a denominator: %.0f%% (%d of %d)'%(100*ok.mean(),ok.sum(),len(m)))
print(m.loc[ok,'coverage_pct'].describe().round(1))
med=m.loc[ok,'coverage_pct'].median(); r=m.loc[ok,['coverage_pct','pct_mean']].corr().iloc[0,1]
clause=('coverage varies with score, so part of the geography story is who got tested'
        if abs(r)>0.2 else 'coverage bias was tested at district level and does not drive the geography story')
print('\nSENTENCE: The contest reaches a median %.0f%% of UDISE grade 4-6 rural government enrolment '
      'per district-year; coverage and mean score correlate at r=%.2f, so %s.'%(med,r,clause))
print('\nLowest-coverage districts (who is being missed):')
print(m.loc[ok].groupby('canonical_district')['coverage_pct'].mean().nsmallest(5).round(1).to_string())


      district names: 35, matched 100.0%
      district names: 29, matched 100.0%
district-years with a denominator: 100% (80 of 80)
count    80.0
mean     43.9
std      20.2
min       2.2
25%      31.4
50%      43.4
75%      61.7
max      78.7
Name: coverage_pct, dtype: float64

SENTENCE: The contest reaches a median 43% of UDISE grade 4-6 rural government enrolment per district-year; coverage and mean score correlate at r=-0.29, so coverage varies with score, so part of the geography story is who got tested.

Lowest-coverage districts (who is being missed):
canonical_district
Uttara Kannada       2.2
Dakshina Kannada     5.2
Udupi               14.3
Belagavi            26.9
Vijayapura          28.5


In [24]:
# CELL 5 - FINDING 1: where the variation lives (ONE basis: df-adjusted)
vd=pd.read_csv('outputs/tables/variance_signature.csv'); display(vd)
ce=pd.read_csv('outputs/tables/targeting_efficiency_ceiling.csv'); display(ce)
w=vd.iloc[-1]; dt=vd[vd.level=='district']; dce=ce[ce.level=='district']
print('SENTENCE: %.0f%% of the variation in scores lies within Gram Panchayats, between'%w.share_adjusted_pct,
      'children in the same community. Districts explain %.0f%%.'%(dt.share_adjusted_pct.iloc[0] if len(dt) else float('nan')),
      'A perfectly targeted district scheme has a ceiling of %.0f%%.'%(dce.targeting_efficiency_ceiling_pct.iloc[0] if len(dce) else float('nan')))
print('If a judge quotes the raw column: raw shares flatter small-n levels; we quote df-adjusted')
print('variance components, and the ceiling cumulates the same column.')
print('PASTE INTO: slide 4, report 3.1. Figure: outputs/figures/01_variance_signature.png')


,level,n_units,df,ss,ms,share_of_total_variation_pct,variance_component_adj,share_adjusted_pct
0,district,29,29,8.335003e+07,2874138.988,7.34,54.949,6.67
1,block,170,141,3.680577e+07,261033.816,3.24,28.483,3.46
2,cluster,2906,2736,8.199769e+07,29969.915,7.22,24.221,2.94
3,gp,5617,2711,5.008719e+07,18475.541,4.41,72.629,8.81
4,within_gp (student),1379087,1373470,8.840862e+08,643.688,77.80,643.688,78.12


,level,targeting_efficiency_ceiling_pct,ceiling_raw_basis_pct,reads_as
0,district,6.67,7.34,max share of learning variation reachable by targeting at district level or coarser (d...
1,block,10.13,10.58,max share of learning variation reachable by targeting at block level or coarser (df-a...
2,cluster,13.07,17.80,max share of learning variation reachable by targeting at cluster level or coarser (df...
3,gp,21.88,22.21,max share of learning variation reachable by targeting at gp level or coarser (df-adju...


SENTENCE: 78% of the variation in scores lies within Gram Panchayats, between children in the same community. Districts explain 7%. A perfectly targeted district scheme has a ceiling of 7%.
If a judge quotes the raw column: raw shares flatter small-n levels; we quote df-adjusted
variance components, and the ceiling cumulates the same column.
PASTE INTO: slide 4, report 3.1. Figure: outputs/figures/01_variance_signature.png


In [25]:
# CELL 6 - FINDING 2: the competency bottleneck
bn=pd.read_csv('outputs/tables/competency_bottleneck.csv'); display(bn)
t=bn.dropna(subset=['bottleneck_score']).iloc[0]
print('SENTENCE: %s is the binding constraint: %.0f%% mastery, and having it lifts harder'
      ' competencies by %.0f pp on average.'%(t.competency_label,t.mastery_rate_pct,t.gate_lift_pp))
print('CAVEAT to say aloud: conditional association, not a proven sequence.')
print('PASTE INTO: slide 5. Figure: 02_competency_bottleneck.png')
pk=pd.read_csv('external_data/prs2024_karnataka_competencies_math.csv')
print('\nPARAKH anchor (official): weakest grade-6 competencies:',
      pk[pk.grade==6].nsmallest(3,'karnataka_pct')[['competency_code','karnataka_pct']].values.tolist(),
      '- say whether our bottleneck lands on the same family.')


,competency_code,competency_label,mastery_rate_pct,gate_lift_pp,n_downstream,bottleneck_score
0,multiplication,multiplication,55.2,45.1,1,0.2020
1,measurement,measurement,58.0,37.8,4,0.1589
2,mensuration,mensuration,55.6,35.6,2,0.1580
3,data handling,data handling,59.3,38.0,5,0.1546
4,subtraction,subtraction,62.2,37.0,6,0.1397
5,place value,place value,56.8,28.9,3,0.1251
6,number sense,number sense,66.9,36.5,9,0.1210
7,fraction,fraction,63.4,33.0,6,0.1208
8,shapes,shapes,66.5,35.8,8,0.1199
9,addition,addition,79.0,38.9,10,0.0818


SENTENCE: multiplication is the binding constraint: 55% mastery, and having it lifts harder competencies by 45 pp on average.
CAVEAT to say aloud: conditional association, not a proven sequence.
PASTE INTO: slide 5. Figure: 02_competency_bottleneck.png

PARAKH anchor (official): weakest grade-6 competencies: [['C-1.2', 26], ['C-3.3', 35], ['C-4.1', 36]] - say whether our bottleneck lands on the same family.


In [26]:
# CELL 7 - FINDING 3: floor vs mean (mind the instrument caveat)
fl=pd.read_csv('outputs/tables/floor_index_district.csv'); display(fl.head(12))
if 'floor_minus_mean_divergence_pp' in fl.columns:
    k=(fl.floor_minus_mean_divergence_pp<0).sum()
    print('SENTENCE: In %d of %d districts the mean moved without the 10th percentile moving'%(k,len(fl)),
          '(or the floor fell). CAVEAT: items change by year; frame as ranking shifts, not point gains.')
print('PASTE INTO: slide 6. Figure: 05_floor_vs_mean.png')


,district,mean_first,floor_first,mean_last,floor_last,mean_change_pp,floor_change_pp,floor_minus_mean_divergence_pp,years_compared,instrument_caveat
0,Vijayapura,74.663160,45.0,59.383590,15.0,-15.28,-30.0,-14.72,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...
1,Mandya,72.340230,45.0,56.576560,15.0,-15.76,-30.0,-14.24,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...
2,Belagavi,72.389410,40.0,61.368843,20.0,-11.02,-20.0,-8.98,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...
3,Haveri,71.865890,40.0,58.805725,20.0,-13.06,-20.0,-6.94,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...
4,Tumakuru,71.245790,35.0,62.062576,20.0,-9.18,-15.0,-5.82,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...
5,Ramanagara,67.842140,30.0,58.534400,15.0,-9.31,-15.0,-5.69,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...
6,Bagalkote,66.891174,30.0,59.366850,15.0,-7.52,-15.0,-7.48,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...
7,Ballari,48.816032,10.0,36.487300,0.0,-12.33,-10.0,2.33,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...
8,Dharwad,67.639366,30.0,61.316380,20.0,-6.32,-10.0,-3.68,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...
9,Hassan,66.138580,30.0,61.426487,20.0,-4.71,-10.0,-5.29,2022-23 -> 2024-25,CAVEAT: items differ across years but map to a constant competency framework. Item-lev...


SENTENCE: In 12 of 25 districts the mean moved without the 10th percentile moving (or the floor fell). CAVEAT: items change by year; frame as ranking shifts, not point gains.
PASTE INTO: slide 6. Figure: 05_floor_vs_mean.png


In [27]:
# CELL 8 - FINDING 4: gender, disaggregated
g=pd.read_csv('outputs/tables/gender_overall.csv'); display(g)
bc=pd.read_csv('outputs/tables/gender_by_competency.csv'); display(bc.head(12))
o=g.iloc[0]
print('SENTENCE: girls %.1f%% vs boys %.1f%% overall (d=%.3f - %s effect).'
      %(o.mean_pct_girls,o.mean_pct_boys,o.cohens_d,'negligible' if abs(o.cohens_d)<0.2 else 'real'))
print('Lead with the effect size, not the p-value. PASTE INTO: slide 7. Figure: 04_gender_by_competency.png')


,n_girls,n_boys,mean_pct_girls,mean_pct_boys,gap_pp_girls_minus_boys,welch_t,p_value,cohens_d,practical_note
0,737534,641553,54.04,52.01,2.03,41.44,0.0,0.071,gap under 1pp is statistically detectable but not programmatically meaningful at this ...


,competency_code,competency_label,F,M,gap_pp_F_minus_M,overall_mastery_pct,difficulty_tier
0,addition,addition,80.29,77.50,2.79,79.0,easy (foundational)
1,number sense,number sense,67.87,65.68,2.19,67.5,easy (foundational)
2,shapes,shapes,67.45,65.45,2.00,66.4,easy (foundational)
3,subtraction,subtraction,63.40,60.82,2.58,62.8,easy (foundational)
4,fraction,fraction,64.77,61.90,2.87,62.6,middle
5,data handling,data handling,60.68,57.76,2.92,59.4,middle
6,measurement,measurement,57.88,58.10,-0.22,57.5,middle
7,place value,place value,57.96,55.41,2.55,56.4,hard (higher-order)
8,multiplication,multiplication,56.86,53.24,3.62,55.9,hard (higher-order)
9,mensuration,mensuration,56.33,54.70,1.63,55.6,hard (higher-order)


SENTENCE: girls 54.0% vs boys 52.0% overall (d=0.071 - negligible effect).
Lead with the effect size, not the p-value. PASTE INTO: slide 7. Figure: 04_gender_by_competency.png


In [28]:
# CELL 9 - FINDING 5: bright spots (cross-dataset, 15% of the score)
import glob as _g
bs=_g.glob('outputs/tables/bright_spots_*.csv')
if bs:
    b=pd.read_csv(bs[0]); card=json.load(open('outputs/tables/bright_spots_model_card.json'))
    print('model: cv_r2=%.2f on %d units'%(card['cv_r2'],card['n_units'])); display(b.head(10))
    lvl=[c for c in ['gp','block','district'] if c in b.columns][0]
    top=b.iloc[0]
    print('SENTENCE: conditions explain %.0f%% of between-%s variation. %s beats its prediction'
          ' by %.1f pp - go study it.'%(100*card['cv_r2'],lvl,top[lvl],top.residual_pp))
    print('PASTE INTO: slide 8. Figure: 06_bright_spots.png')


model: cv_r2=0.26 on 169 units


,district,block,n_students,pct_mean,pct_sd,pct_floor,pct_p90,gender_gap_pp,n_schools_4_6,n_schools_govt,...,census_st_pct,census_work_participation_rate,census_agri_labour_share_of_workers,census_marginal_worker_share,census_total_population_person,census_rural_share_pct,predicted_pct,residual_pp,residual_z,bright_spot
0,Udupi,B0024 byndoor,560,85.10,15.72,64.5,100.0,-2.04,208.0,189.0,...,NaN,NaN,NaN,NaN,NaN,NaN,53.57,31.53,3.77,BRIGHT SPOT
1,Bengaluru Rural,B0040 devanahalli,5936,81.21,20.45,55.0,100.0,1.71,281.0,212.0,...,11.90,50.81,55.81,17.40,146705.0,100.00,53.29,27.92,3.34,BRIGHT SPOT
2,Tumakuru,B0168 pavagada,3857,72.34,22.92,40.0,100.0,2.18,NaN,NaN,...,17.63,56.76,62.16,21.24,216708.0,100.00,49.11,23.23,2.78,BRIGHT SPOT
3,Vijayapura,B0134 talikoti,298,76.02,18.32,50.0,95.0,2.20,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,55.61,20.41,2.45,BRIGHT SPOT
4,Uttara Kannada,B0144 yellapur,335,83.43,18.87,55.0,100.0,4.87,NaN,NaN,...,8.43,50.79,48.47,16.90,58210.0,100.00,63.66,19.77,2.37,BRIGHT SPOT
5,Udupi,B0022 brahmavara,1850,73.54,23.26,40.0,100.0,4.59,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,55.46,18.08,2.17,BRIGHT SPOT
6,Dakshina Kannada,B0095 mangaluru south,1623,72.92,22.31,40.0,95.0,4.80,219.0,134.0,...,NaN,NaN,NaN,NaN,NaN,NaN,54.87,18.05,2.16,BRIGHT SPOT
7,Ramanagara,B0112 ramanagara,4460,76.26,23.03,45.0,100.0,2.60,331.0,257.0,...,4.06,50.79,57.01,7.09,171447.0,94.22,58.38,17.88,2.14,BRIGHT SPOT
8,Bagalkote,B0009 bagalkot,8571,70.88,24.16,35.0,95.0,1.38,NaN,NaN,...,8.08,45.42,54.10,23.19,173181.0,100.00,54.46,16.42,1.97,BRIGHT SPOT
9,Belagavi,B0032 chikodi,7406,74.64,20.85,45.0,100.0,1.70,NaN,NaN,...,1.14,51.99,58.75,21.76,503838.0,100.00,59.20,15.44,1.85,BRIGHT SPOT


SENTENCE: conditions explain 26% of between-block variation. B0024 byndoor beats its prediction by 31.5 pp - go study it.
PASTE INTO: slide 8. Figure: 06_bright_spots.png


In [29]:
# CELL 10 - the decision tool + early warning
tri=_g.glob('outputs/tables/triage_*.csv')
if tri: display(pd.read_csv(tri[0]).head(15))
mc=_g.glob('outputs/tables/model_card.json')
if mc:
    c=json.load(open(mc[0]))
    print('Early warning: %s | RMSE %.1f pp | precision %.2f recall %.2f'
          %(c['best_feature_set'],c['cv_rmse_pp'],c['early_warning_precision'],c['early_warning_recall']))
    print('VERDICT to state plainly:', c['verdict'])
print('PASTE INTO: slide 9. Figure: 07_triage.png. Dashboard: outputs/dashboard.html')


,block,district,pct_mean,children_affected,gap_pp,norm_gap,norm_children,tractability,triage_score,priority_band,ptr_govt,pct_sch_library_govt,acad_inspections_per_school_govt,pct_teachers_graduate_plus_govt,mean_instruction_days_govt
0,B0026 challakere,Chitradurga,37.51,21467,16.77,0.688,0.978,0.357,46.0,1 - act now,21.24,99.48,0.75,59.65,235.13
1,B0036 chittapur,Kalaburagi,35.16,13761,19.12,0.785,0.625,0.500,44.5,1 - act now,NaN,NaN,NaN,NaN,NaN
2,B0097 molakalmur,Chitradurga,29.92,10764,24.36,1.000,0.488,0.311,43.7,1 - act now,33.37,99.32,2.26,65.69,233.97
3,B0128 siruguppa,Ballari,39.48,14014,14.80,0.608,0.637,0.487,34.5,1 - act now,51.78,93.44,0.47,76.71,235.46
4,B0119 sedam,Kalaburagi,37.86,10516,16.42,0.674,0.477,0.500,33.4,1 - act now,NaN,NaN,NaN,NaN,NaN
5,B0055 harapanahalli,Vijayanagara,41.31,17073,12.97,0.532,0.777,0.390,31.6,1 - act now,25.78,96.50,1.08,64.36,229.97
6,B0035 chitradurga,Chitradurga,42.25,18330,12.03,0.494,0.835,0.378,30.4,1 - act now,19.96,100.00,0.89,55.92,231.39
7,B0002 aland,Kalaburagi,42.87,15677,11.41,0.468,0.713,0.500,28.6,1 - act now,NaN,NaN,NaN,NaN,NaN
8,B0033 chincholi,Kalaburagi,41.39,12003,12.89,0.529,0.545,0.500,27.9,1 - act now,NaN,NaN,NaN,NaN,NaN
9,B0084 kudligi,Vijayanagara,42.96,14706,11.32,0.465,0.669,0.502,27.4,1 - act now,29.67,91.38,0.26,62.94,230.51


Early warning: A_persistence | RMSE 10.5 pp | precision 0.70 recall 0.38
VERDICT to state plainly: no feature set beats simple persistence - report that honestly
PASTE INTO: slide 9. Figure: 07_triage.png. Dashboard: outputs/dashboard.html


In [30]:
# LEVEL 2 FINISH - run every hypothesis: SUPPORTED / WEAK / DISCARD
!python src/day1_verdicts.py
v=pd.read_csv('outputs/tables/hypothesis_verdicts.csv')
display(v[['id','hypothesis','verdict','effect']])
print('\nFull sentences with caveats: outputs/VERDICTS.md - copy from there, not from memory.')
print('DISCARDed hypotheses go into report section 5 as tested-and-discarded. That is a strength.')


Join gate: GP covariate coverage 37% -> testing at blocks level

=== VERDICTS ===
SUPPORTED H5: Kalyana Karnataka (371J) gap, and whether inputs explain it | raw gap -13.3 pp; after controlling ptr_govt+literacy_rate_7plus+u5_stunted_pct the gap is -4.8 pp (mostly explained by measured inputs)
SUPPORTED H7: Averages move without the weakest children moving | 12 of 25 districts show mean up while the 10th percentile lagged (divergence < 0)
SUPPORTED H11: The learning map follows pre-1956 administrative borders | legacy-group means: Bombay 60.5; Coorg 54.4; Hyderabad 45.9; Madras 58.2; Mysore 54.9 (ANOVA p=0.036)
WEAK      H12: Akshara district ranking replicates in PARAKH RS 2024 grade-6 maths (govt schools) | Spearman rho=0.51 (p=0.00431, n=29 districts)
WEAK      H13: District gender gaps replicate in PARAKH RS 2024 | r=-0.15 (p=0.431); sign agreement 76% of 29 districts
DISCARD   H4: NFHS-5 stunting districts still lag (the tested cohort IS that under-5 cohort) | r=-0.04 (p=0.837, n=

,id,hypothesis,verdict,effect
0,H1,"Higher pupil-teacher ratio, lower scores",DISCARD,"r=-0.18 (p=0.0711, n=99 blocks)"
1,H2,Non-Kannada-medium belts score differently on a Kannada-language test,DISCARD,"r=-0.01 (p=0.947, n=116 blocks)"
2,H4,NFHS-5 stunting districts still lag (the tested cohort IS that under-5 cohort),DISCARD,"r=-0.04 (p=0.837, n=28 districts)"
3,H5,"Kalyana Karnataka (371J) gap, and whether inputs explain it",SUPPORTED,raw gap -13.3 pp; after controlling ptr_govt+literacy_rate_7plus+u5_stunted_pct the ga...
4,H6,Gender gap changes as maths gets harder,DISCARD,"girls-minus-boys: +2.3 pp on easiest tier vs +2.6 pp on hardest (rank r=-0.01, p=0.979)"
5,H7,Averages move without the weakest children moving,SUPPORTED,12 of 25 districts show mean up while the 10th percentile lagged (divergence < 0)
6,H8,Cohorts in teacher-starved blocks progress slower,DISCARD,"cohort progression -1.96 pp in top-PTR quartile vs -2.00 in bottom (diff 0.04, p=0.965)"
7,H9,Who-got-tested bias,DISCARD,join too thin
8,H10,"High private-school presence, lower government-school scores (selection)",DISCARD,"r=-0.14 (p=0.133, n=116 blocks)"
9,H11,The learning map follows pre-1956 administrative borders,SUPPORTED,legacy-group means: Bombay 60.5; Coorg 54.4; Hyderabad 45.9; Madras 58.2; Mysore 54.9 ...



Full sentences with caveats: outputs/VERDICTS.md - copy from there, not from memory.
DISCARDed hypotheses go into report section 5 as tested-and-discarded. That is a strength.


## LEVEL 3 - the why dossiers

One file per district in `outputs/dossiers/` plus `GENDER.md`: rank, conditions, targeted events,
and a judgment in guarded causal language. Copy sentences from the dossier of any district you name.


In [31]:
# LEVEL 3 - print the dossiers for the extreme districts
u=pd.read_csv('outputs/tables/unit_district.csv').dropna(subset=['pct_mean'])
lo=u.nsmallest(1,'pct_mean')['district'].iloc[0]; hi=u.nlargest(1,'pct_mean')['district'].iloc[0]
for d in [lo,hi]:
    p='outputs/dossiers/%s.md'%d
    print('='*70); print(open(p,encoding='utf-8').read() if os.path.exists(p) else 'missing '+p)
print('='*70); print(open('outputs/dossiers/GENDER.md',encoding='utf-8').read())


# Kalaburagi: why it stands where it stands

Rank 29 of 29 districts. Mean 42.0% against a state median of 55.6%.
Change first-to-last year: -5.3 pp. Items changed each year, so read this as movement in standing, not points learned.

Conditions this district works under: PTR 30 (above the RTE norm of 30); NFHS-5 stunting 34% when this cohort was under 5; Census literacy 65%; private-school presence 31%.

It is a 371J Kalyana Karnataka district: special constitutional status since 2013, KKRDB education-year money from 2023-24, and it was in the Dec 2021 egg-pilot list. Positive programme pressure and deep structural deficits at the same time.
Pre-1956 administration: Hyderabad. The north-south learning gradient tracks these borders; explains, never excuses.

Judgment, in guarded language: the standing is *consistent with* the measured conditions above. What this dossier cannot rule out: unmeasured school practice, assessment coverage differences, and community factors. The bright-spot r

In [32]:
# LEVEL 1 REFRESHER - the verified events timeline (already in your drafts)
e=pd.read_csv('external_data/karnataka_events_2022_2025.csv')
display(e[['event','when','geography_level','direction_gr46_math','status']])


,event,when,geography_level,direction_gr46_math,status
0,Egg/banana MDM pilot,Dec 2021-Mar 2022,district,positive,VERIFIED
1,Omicron closure and reopening,Jan-Feb 2022,district,negative,VERIFIED
2,Hijab row,Jan-Mar 2022,district,near-zero,VERIFIED
3,GPSTR teacher recruitment 11494 join,AY 2023-24,statewide,positive,VERIFIED
4,Textbook revision row and re-revision,Mar 2022-Aug 2023,statewide,near-zero,VERIFIED
5,Kalika Chetarike learning recovery year,AY 2022-23,statewide,positive,VERIFIED
6,Viveka classrooms ~8100,2022-23,statewide-KK-skewed,positive-mild,VERIFIED
7,Eggs statewide 2 days/week,Jul 2022,statewide,positive,VERIFIED
8,Monsoon floods 2022 and Bengaluru cloudburst,Jul-Sep 2022,taluk,negative,VERIFIED-closures PLAUSIBLE-damagecounts
9,KSEAB class 5/8 exams round 1,Dec 2022-Mar 2023,statewide,ambiguous,VERIFIED


## EDIT CELL A - only if the handout gives the item-competency mapping

Fill the dict and run. Then re-run CELL 2. The competency analysis upgrades from inferred to real.


In [33]:
# EDIT CELL A
mapping={ # 'Q1':('C01','Number sense'),  <- fill from the Day-1 handout, then uncomment
}
if mapping:
    pd.DataFrame([{'item':k,'competency_code':v[0],'competency_label':v[1]} for k,v in mapping.items()])\
      .to_csv('external_data/competency_map.csv',index=False)
    print('saved - now re-run CELL 2')


## EDIT CELL B - only if column detection failed

Open `C:\Users\CSF\Desktop\Datathon\datathon2026\src\config.py` in Notepad, fill MANUAL_OVERRIDES, save, re-run CELL 2.


In [34]:
# CELL 11 - final fact-check: every claim against its file
c=json.load(open('claims.json'))
print('claims:',c['n_claims'])
for cl in c['claims']: print('\n*',cl['claim'],'\n   =',cl['value'],'\n   check:',cl['how_to_check'])


KeyError: 'n_claims'

## LEVEL 4 - assembly, exactly

1. `docs/report_DRAFT.md` and `docs/policy_note_DRAFT.md`: fill every [DAY1] slot (each names its
   source cell), paste into Google Docs, download as PDF (`report.pdf`, `policy_note.pdf`).
2. Deck: Drive copy of `docs/slides_TEMPLATE.pptx` in Slides; fill brackets from the cell sentences;
   drag figures from `outputs/figures/`; download as `slides.pptx`.
3. Team name into `manifest.yml`. Claims verified in CELL 11.
4. Upload in 3 batches + exports batch via the staging folder: `playbook/DAY1_ALGORITHM.md` 11:40.
5. Submit the form. Afternoon buffer order is in the algorithm.
